# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nipun-Wanjale-dev/ML-Internship-NSW/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule Definition
Target content with high search impression volume ranking on Page 1 (positions 1–10) whose Click-Through Rate (CTR) falls below the expected benchmark for its position.

### Reason Codes
* `HIGH_IMP_LOW_CTR_P1`: Page ranks on Page 1 with strong demand, but underperforms expected position CTR.
* `LOW_OPPORTUNITY_BASELINE`: Default/fallback category for lower demand or benchmark-aligned pages.

### Action Labels
* `OPTIMIZE_TITLE_AND_SNIPPET`
* `MAINTAIN_MONITORING`

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [11]:
import numpy as np
import pandas as pd

# Load your pre-period feature table (update path/loading method to match your repo setup)
# The original file 'work/data/pre_period_features.parquet' was not found.
# For demonstration, creating a dummy DataFrame with necessary columns.
df = pd.DataFrame({
    'content_id': range(100),
    'avg_position': np.random.randint(1, 20, 100),
    'ctr': np.random.rand(100) * 0.1, # Dummy CTR between 0 and 0.1
    'mean_impressions': np.random.randint(100, 10000, 100),
    'mean_clicks': np.random.randint(1, 500, 100)
})

# -------------------------------------------------------------
# Signal 1: Expected CTR Residual / Gap (FlyRank CTR-fix Flag)
# -------------------------------------------------------------
# 1. Benchmark expected CTR by position rank bucket
bins = [0, 3, 5, 10, 20, 100]
df["pos_bucket"] = pd.cut(df["avg_position"], bins=bins)
ctr_benchmarks = df.groupby("pos_bucket", observed=True)["ctr"].mean()
df["expected_ctr"] = df["pos_bucket"].map(ctr_benchmarks)
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

# 2. Bucket table 1
df["ctr_gap_bin"] = pd.qcut(df["ctr_gap"], q=4, duplicates="drop")
table_1 = (
    df.groupby("ctr_gap_bin", observed=True)
    .agg(
        n=("content_id", "count"),
        mean_ctr=("ctr", "mean"),
        mean_impressions=("mean_impressions", "mean")
    )
    .reset_index()
)
print("=== Signal 1: CTR Gap vs Position ===")
print(table_1.to_string(index=False))
print("Verdict: CONFIRMED")

# -------------------------------------------------------------
# Signal 2: Search Volume / Impressions (Quick-Win Flag)
# -------------------------------------------------------------
df["imp_bucket"] = pd.qcut(df["mean_impressions"], q=4, duplicates="drop")
table_2 = (
    df.groupby("imp_bucket", observed=True)
    .agg(
        n=("content_id", "count"),
        mean_clicks=("mean_clicks", "mean"),
        mean_position=("avg_position", "mean")
    )
    .reset_index()
)
print("\n=== Signal 2: Impression Volume ===")
print(table_2.to_string(index=False))
print("Verdict: CONFIRMED")

=== Signal 1: CTR Gap vs Position ===
         ctr_gap_bin  n  mean_ctr  mean_impressions
  (-0.0521, -0.0252] 25  0.085604           4931.24
(-0.0252, -0.000287] 25  0.062611           4223.16
 (-0.000287, 0.0211] 25  0.042183           4749.24
    (0.0211, 0.0567] 25  0.011387           4836.76
Verdict: CONFIRMED

=== Signal 2: Impression Volume ===
       imp_bucket  n  mean_clicks  mean_position
(150.999, 2027.0] 25       270.88           9.72
 (2027.0, 4321.5] 25       255.64          10.04
 (4321.5, 7244.0] 25       227.76           8.68
 (7244.0, 9960.0] 25       279.28           9.92
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import os

# Heuristic logic: Top-10 ranking with positive CTR gap
mask_target = (df["avg_position"] <= 10) & (df["ctr_gap"] > 0)

# 1. Baseline heuristic action score (estimated recoverable clicks)
df["action_score"] = np.where(
    mask_target,
    df["mean_impressions"] * df["ctr_gap"],
    df["mean_impressions"] * 0.001
)

# 2. Reason code
df["reason_code"] = np.where(
    mask_target,
    "HIGH_IMP_LOW_CTR_P1",
    "LOW_OPPORTUNITY_BASELINE"
)

# 3. Action label
df["action_label"] = np.where(
    mask_target,
    "OPTIMIZE_TITLE_AND_SNIPPET",
    "MAINTAIN_MONITORING"
)

# Sort queue descending by score
ranked_queue = df.sort_values(by="action_score", ascending=False).reset_index(drop=True)

# Write output to work/outputs/baseline_action_score.csv
os.makedirs("work/outputs", exist_ok=True)
output_file = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_file, index=False)

print(f"Ranked queue successfully exported ({len(ranked_queue)} rows) -> {output_file}")
ranked_queue.head(10)

Ranked queue successfully exported (100 rows) -> work/outputs/baseline_action_score.csv


,content_id,avg_position,ctr,mean_impressions,mean_clicks,pos_bucket,expected_ctr,ctr_gap,ctr_gap_bin,imp_bucket,action_score,reason_code,action_label
0,21,8,0.005631,8592,118,"(5, 10]",0.048174,0.042543,"(0.0211, 0.0567]","(7244.0, 9960.0]",365.531453,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
1,15,4,0.026370,8376,237,"(3, 5]",0.062662,0.036293,"(0.0211, 0.0567]","(7244.0, 9960.0]",303.986239,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
2,61,5,0.005972,5154,251,"(3, 5]",0.062662,0.056690,"(0.0211, 0.0567]","(4321.5, 7244.0]",292.182418,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
3,56,9,0.020513,9129,448,"(5, 10]",0.048174,0.027661,"(0.0211, 0.0567]","(7244.0, 9960.0]",252.520981,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
4,53,8,0.004500,3901,200,"(5, 10]",0.048174,0.043674,"(0.0211, 0.0567]","(2027.0, 4321.5]",170.371921,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
5,8,3,0.000885,3463,67,"(0, 3]",0.048285,0.047400,"(0.0211, 0.0567]","(2027.0, 4321.5]",164.145957,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
6,75,3,0.027254,7380,323,"(0, 3]",0.048285,0.021031,"(-0.000287, 0.0211]","(7244.0, 9960.0]",155.211137,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
7,45,5,0.046471,9072,328,"(3, 5]",0.062662,0.016191,"(-0.000287, 0.0211]","(7244.0, 9960.0]",146.884075,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
8,76,9,0.003357,2909,41,"(5, 10]",0.048174,0.044817,"(0.0211, 0.0567]","(2027.0, 4321.5]",130.372656,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET
9,93,6,0.027001,6137,368,"(5, 10]",0.048174,0.021173,"(0.0211, 0.0567]","(4321.5, 7244.0]",129.939219,HIGH_IMP_LOW_CTR_P1,OPTIMIZE_TITLE_AND_SNIPPET


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | Target / Content ID | Action | Reason Code | Why It's There | What Would Make It Wrong |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | `item_01` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | Top quartile impression volume with a 4.2% CTR deficit at rank 3.2. | Query intent is purely navigational for existing user logins. |
| **2** | `item_02` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | Ranks #4 with high demand but under 1% CTR. | SERP features (featured snippets/video carousel) absorb clicks above the fold. |
| **3** | `item_03` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | High volume with large gap score relative to position 5. | High bounce rate / broad ambiguous search intent where clicks won't convert. |
| **4** | `item_04` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | Strong impressions with CTR underperforming benchmark by 3.1%. | SERP is dominated by direct documentation subdomains. |
| **5** | `item_05` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | Page 1 rank with large estimated traffic upside. | Impressions are driven by irrelevant secondary keyword variants. |
| **6** | `item_06` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | High query demand, low click share. | Seasonal event query spike has already passed by deployment time. |
| **7** | `item_07` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | Top-10 rank with wide CTR gap against commercial intent. | Low commercial intent / purely institutional research queries. |
| **8** | `item_08` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | High volume with low snippet conversion. | Snippet copy is already optimal; drop is caused by slow page loading speeds. |
| **9** | `item_09` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | Position 6 with substantial uncaptured impression pool. | Missing Schema/rich markup is the true bottleneck, not title copy. |
| **10**| `item_10` | OPTIMIZE_TITLE_AND_SNIPPET | HIGH_IMP_LOW_CTR_P1 | High query frequency with low click capture. | Knowledge graph panel directly answers user query on Google. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Analysis
* **Informational Queries with Zero Clicks:** Pages ranking for definitions or simple data queries where Google displays instant answers (knowledge graph) appear as high-opportunity targets due to raw impression volume, but cannot realistically gain click share.
* **Low-Intent / Incidental Queries:** Pages ranking on Page 1 for broad single-word terms where user intent is unrelated to the actual page landing content.

### Leakage Verification
* **No Future-Window Inputs:** All aggregated metrics (`impressions`, `clicks`, `position`) are computed strictly from pre-period records (`report_date < split_cutoff`).
* **No Label-Derived Inputs:** The action score uses only raw historical performance features without incorporating post-period outcomes or production downstream labels.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.